<a href="https://colab.research.google.com/github/ngtan369/Hybrid-Image-Classification/blob/main/notebooks/ex3_imageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("Hello Machine Learning Course !!")

Hello Machine Learning Course !!


## Download Source Code
 ### github repo: https://github.com/ngtan369/Hybrid-Image-Classification

In [3]:
!rm -rf /content/repo
!git clone https://github.com/ngtan369/Hybrid-Image-Classification /content/repo
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")
print("Added /content/repo to PYTHONPATH")

Cloning into '/content/repo'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 115 (delta 36), reused 70 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 42.85 KiB | 1.65 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/repo
Added /content/repo to PYTHONPATH


# Download data
## this part git data set from Kaggle, author: jcoral02, data set name: inriaperson

In [4]:
# Cell 1: Tải dataset từ Kaggle (kagglehub)
# Cài đặt và tải dataset INRIA (chạy trên Colab
%pip install kagglehub -q
import kagglehub, os
print("Downloading INRIA Person from Kaggle...")
dataset_path = kagglehub.dataset_download("jcoral02/inriaperson")
print("Done. dataset_path =", dataset_path)

100%|██████████| 582M/582M [00:03<00:00, 173MB/s]

Extracting files...


Done. dataset_path = /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


# Traditional Machine Learning Pipeline

Machine Learning Traditional - Full Pipeline (EDA -> Train -> Eval)

Vai trò: Đây là cách tiếp cận 'Hybrid'. Sử dụng sức mạnh của Deep Learning (ResNet50) để làm bộ trích xuất đặc trưng (Feature Extractor), nhưng dùng các thuật toán học máy cổ điển (SVM, Logistic Regression) để phân loại.

Ưu điểm trình bày: Cho thấy khả năng linh hoạt (config chọn model/classifier). Đây là quy trình tiết kiệm tài nguyên nhưng hiệu quả rất cao trên tập dữ liệu nhỏ.

Nội dung: EDA chi tiết, trích xuất đặc trưng, lưu file .npy, huấn luyện SVM và đánh giá qua Confusion Matrix.

In [ ]:

import os, sys, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- Configuration for Flexibility ---
CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "pretrained_model": "resnet50", # Options: vgg16, resnet50, efficientnet
    "classifier": "svm",           # Options: svm, logistic_regression, random_forest
    "repo_path": "/content/repo"
}

# Fix: Ensure the repository path is correctly handled and check for module existence
repo_root = Path(CONFIG["repo_path"]).resolve()
if repo_root.exists():
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # Check if 'modules' exists inside the repo
    if not (repo_root / "modules").exists():
        print(f"Warning: 'modules' folder not found in {repo_root}. Contents: {os.listdir(repo_root)}")
else:
    print(f"Error: Repo path {repo_root} does not exist. Please check the git clone cell.")

try:
    from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk
    print(f"Pipeline configured with {CONFIG['pretrained_model']} and {CONFIG['classifier']}")
except ModuleNotFoundError as e:
    print(f"Still failing to import: {e}")
    print("Try running: !ls -R /content/repo to see the folder structure.")

Still failing to import: No module named 'modules.image_utils'
Try running: !ls -R /content/repo to see the folder structure.


## Import Library necessary

In [9]:
import os, sys, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- Configuration for Flexibility ---
CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "pretrained_model": "resnet50", # Options: vgg16, resnet50, efficientnet
    "classifier": "svm",           # Options: svm, logistic_regression, random_forest
    "repo_path": "/content/repo"
}

# Fix: Ensure the repository path is correctly handled and check for module existence
repo_root = Path(CONFIG["repo_path"]).resolve()
if repo_root.exists():
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
else:
    print(f"Error: Repo path {repo_root} does not exist. Please check the git clone cell.")

try:
    # Updated based on actual folder structure (ml_utils and dl_utils)
    from modules.ml_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk
    print(f"Pipeline configured with {CONFIG['pretrained_model']} and {CONFIG['classifier']}")
except ModuleNotFoundError as e:
    print(f"Still failing to import: {e}")
    print("Try running: !ls -R /content/repo to see the folder structure.")

ImportError: cannot import name 'load_and_preprocess_data' from 'modules.ml_utils' (/content/repo/modules/ml_utils.py)

In [8]:
!ls -R /content/repo

/content/repo:
features  modules  notebooks  README.md  reports  requirement.txt

/content/repo/features:
bai3.npy

/content/repo/modules:
dl_utils.py  ml_utils.py

/content/repo/notebooks:
ex3_imageData.ipynb

/content/repo/reports:
report.pdf


In [11]:
with open('/content/repo/modules/ml_utils.py', 'r') as f:
    print(f.read())

import os
from pathlib import Path
import numpy as np
from PIL import Image
from skimage import color, transform, feature

def load_image_paths(root_dir):
    root = Path(root_dir)
    paths = [p for p in root.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')]
    return sorted(paths)

def load_and_resize_images(paths, image_size=(128,128)):
    imgs = []
    for p in paths:
        img = Image.open(p).convert('RGB')
        img = img.resize(image_size)
        imgs.append(np.array(img))
    return np.stack(imgs, axis=0)

def hog_features_from_images(X_images, pixels_per_cell=(16,16), cells_per_block=(2,2), orientations=9, resize=None):
    feats = []
    for img in X_images:
        if resize is not None:
            img = transform.resize(img, resize, anti_aliasing=True)
            img = (img * 255).astype('uint8')
        gray = color.rgb2gray(img)
        h = feature.hog(gray, orientations=orientations, pixels_per_cell=pixels_per_cell,
                        cells_per_blo

## Setup & Data Loading

In [ ]:
if isinstance(dataset_path, str) and dataset_path.endswith('.zip'):
    extract_dir = '/content/inriaperson'
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(dataset_path,'r') as z: z.extractall(extract_dir)
    dataset_root = extract_dir
else:
    dataset_root = dataset_path

def find_image_root(root):
    p = Path(root)
    if not p.exists(): return str(p)
    for child in p.iterdir():
        if child.is_dir() and any(child.glob('**/*.jpg')): return str(p)
    for sub in p.rglob('*'):
        if sub.is_dir() and any(sub.glob('*.jpg')): return str(sub)
    return str(p)

dataset_for_loader = find_image_root(dataset_root)
print(f"Dataset root identified: {dataset_for_loader}")

## Preprocessing & EDA

In [12]:
from modules.ml_utils import load_image_paths, load_and_resize_images

image_size = CONFIG["image_size"]

# Manually implement the data loading using available utilities
print(f"Searching for images in: {dataset_for_loader}")
class_names = sorted([d.name for d in Path(dataset_for_loader).iterdir() if d.is_dir()])

X_list = []
y_list = []

for i, cls in enumerate(class_names):
    paths = load_image_paths(Path(dataset_for_loader) / cls)
    imgs = load_and_resize_images(paths, image_size=image_size)
    X_list.append(imgs)
    y_list.append(np.full(len(imgs), i))

X_raw = np.vstack(X_list)
y_raw = np.concatenate(y_list)

print(f"Loaded {len(X_raw)} images across {len(class_names)} classes.")

# --- 2.1. Label Distribution ---
unique, counts = np.unique(y_raw, return_counts=True)
label_dist = dict(zip([class_names[i] for i in unique], counts))

print("--- Label Distribution ---")
for cls, count in label_dist.items():
    print(f"{cls}: {count} images")

# --- 2.3. Visualization ---
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(x=list(label_dist.keys()), y=list(label_dist.values()))
plt.title("Label Distribution")

plt.subplot(1, 2, 2)
ch_means = X_raw.mean(axis=(0, 1, 2))
plt.bar(['Red', 'Green', 'Blue'], ch_means, color=['red', 'green', 'blue'])
plt.title("Average Pixel Intensity")
plt.show()

NameError: name 'dataset_for_loader' is not defined

## Feature Extraction (Transfer Learning features)

In [ ]:
model_name = CONFIG['pretrained_model']
X_features = extract_features_pretrained(X_raw, model_name=model_name, batch_size=CONFIG['batch_size'], pooling='avg')
print(f"Features extracted using {model_name}. Shape: {X_features.shape}")

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y_raw, test_size=0.2, random_state=42, stratify=y_raw)


## Model Training (SVM)

In [ ]:
clf_type = CONFIG['classifier']
print(f"Training {clf_type.upper()} model...")

if clf_type == 'svm':
    clf = SVC(kernel='linear', C=1.0, probability=True)
elif clf_type == 'logistic_regression':
    clf = LogisticRegression(max_iter=1000)
elif clf_type == 'random_forest':
    clf = RandomForestClassifier(n_estimators=100)

clf.fit(X_train, y_train)

## Evaluation


In [ ]:
y_pred = clf.predict(X_test)
display(Markdown(f"### Kết quả huấn luyện: {CONFIG['pretrained_model']} + {CONFIG['classifier']}"))
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix ({CONFIG['classifier']})")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

 ## Save results


In [ ]:
prefix = "inria_resnet50_svm"
save_features_to_disk(X_features, y_raw, prefix)
print("Pipeline complete. Features and labels saved with prefix:", prefix)

# Deep Learning Pipeline (End-to-End & Feature Export)

Vai trò: Đây là quy trình 'End-to-End'. Bạn xây dựng một mạng nơ-ron hoàn chỉnh (với base là VGG16) và huấn luyện 'phần đầu' (head) của mạng đó trực tiếp trên ảnh.

Ưu điểm trình bày: Dùng để so sánh hiệu năng. Có thể kết luận xem việc dùng đặc trưng ResNet+SVM (ở phần 1) tốt hơn hay dùng VGG16 huấn luyện trực tiếp (ở phần 2) tốt hơn.

Nội dung: Xây dựng kiến trúc Transfer Learning, huấn luyện với TensorFlow/Keras và lưu đặc trưng VGG16 ra file theo yêu cầu.

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

# 1. Data Loading for DL
img_size = CONFIG["image_size"]
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=CONFIG["batch_size"], validation_split=0.2, subset='training', seed=42)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=CONFIG["batch_size"], validation_split=0.2, subset='validation', seed=42)

# 2. Build Model using VGG16
base_model = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(*img_size, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(*img_size, 3))
x = tf.keras.applications.vgg16.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(len(train_ds.class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 3. Training and Feature Export
print("Training Deep Learning Head...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 4. Save Features to .npy (as requested by assignment)
print("Exporting features to .npy...")
# Re-extract features specifically from the VGG16 base for saving
all_images = []
all_labels = []
for images, labels in train_ds.unbatch().take(-1):
    all_images.append(images.numpy())
    all_labels.append(labels.numpy())

X_array = np.array(all_images)
y_array = np.array(all_labels)

# Extract using the base_model part of our model
X_features_vgg = base_model.predict(tf.keras.applications.vgg16.preprocess_input(X_array))
X_features_vgg = X_features_vgg.reshape(X_features_vgg.shape[0], -1)

np.save('vgg16_features.npy', X_features_vgg)
np.save('vgg16_labels.npy', y_array)

print(f"Deep Learning process complete. Features saved: vgg16_features.npy ({X_features_vgg.shape})")

# So sánh ResNet+SVM với  VGG16

// to do something

In [ ]:
import matplotlib.pyplot as plt

# Giả định lấy kết quả từ quá trình chạy trên
# (Trong thực tế bạn có thể lấy từ history.history hoặc clf.score)

results = {
    "ResNet50 + SVM": clf.score(X_test, y_test),
    "VGG16 (Transfer Learning)": history.history['val_accuracy'][-1]
}

# Hiển thị bảng so sánh
comparison_df = pd.DataFrame(list(results.items()), columns=['Model Combination', 'Accuracy'])
display(comparison_df)

# Vẽ biểu đồ so sánh
plt.figure(figsize=(8, 5))
sns.barplot(x='Model Combination', y='Accuracy', data=comparison_df, palette='magma')
plt.ylim(0, 1.0)
plt.title("So sánh hiệu quả giữa các kiến trúc học sâu")
plt.ylabel("Accuracy (Độ chính xác)")
plt.show()